# Tailwater: risk-constrained RL for irrigation scheduling
CAP 6629 Reinforcement Learning, Fall 2026, final project (Option 2). Aryan Bhagat, FAU.

This notebook reproduces the project from the GitHub repository:

1. clone the code and download the NOAA weather record (Belle Glade, FL, 74 spring seasons),
2. run the 30 unit tests (water conservation, FAO reference values, reward identity, gradients),
3. show the saved results of the full experiment (220 training runs),
4. train a short CVaR-constrained PPO agent on three folds and score it on the held-out fold
   against a baseline tuned on the same training years.

The full experiment takes about an hour on 4 cores (`experiments/run_agents.py`); the short run here takes about a minute.

In [ ]:
!git clone -q https://github.com/Aryanbhagat23/tailwater
%cd tailwater
!pip install -q pytest
import sys; sys.path.insert(0, "src")

In [ ]:
!python scripts/fetch_noaa.py

## 1. Tests

In [ ]:
!python -m pytest -q

## 2. Weather: real seasons have drought years

In [ ]:
from tailwater.data import belle_glade_seasons
ws = belle_glade_seasons()
tot = ws.rain.sum(1)
print(f"{ws.n} seasons {ws.labels[0]}-{ws.labels[-1]}; March-June rain {tot.min():.0f}-{tot.max():.0f} mm, "
      f"CV {tot.std()/tot.mean():.2f}; driest {ws.labels[tot.argmin()]}")
from IPython.display import Image, Markdown, display
display(Image("results/figures/rate_sweep.png", width=800))

## 3. Saved results of the full experiment (held-out years, 5 seeds)

In [ ]:
display(Markdown(open("results/summary.md").read().split("## Tuned baselines")[0]))
display(Image("results/figures/frontier_cvar.png", width=750))
display(Image("results/figures/learning_curves.png", width=900))

## 4. Train a CVaR-constrained PPO agent (short run)
Train on folds 2-4, evaluate on fold 1 (years the agent never sees), and compare with the stage-threshold rule tuned on the same training years for the same reliability floor.

In [ ]:
import time, numpy as np
from tailwater.agents.ppo import PPOConfig, train_ppo
from tailwater.env.irrigation_env import simulate
from tailwater.eval.metrics import summarize
from tailwater.eval.protocol import make_folds, eval_forecast, FORECAST_NOISE, TUNE_FORECAST_SEED
from tailwater.policies.tuning import sweep, select

train, test = make_folds(ws.n)[0]
fc = eval_forecast(ws)
t = time.time()
agent, curve = train_ppo(ws.subset(train), PPOConfig(kappa=2.0, floor=0.90, iterations=80, seed=0), log_every=20)
print(f"trained in {time.time()-t:.0f} s")
rule = select(sweep("stage threshold", ws.subset(train), ws.forecasts(FORECAST_NOISE, TUNE_FORECAST_SEED)[train]),
              "cvar5_yield", 0.90)["policy"]
for name, pol in [("CVaR-PPO (80 iterations)", agent), (rule.label(), rule)]:
    r = simulate(pol, ws.subset(test), fc[test])
    s = summarize(r.water, r.yield_)
    print(f"{name:70s} water {s['water']:5.0f} mm  mean yield {s['mean_yield']:.3f}  CVaR-5% {s['cvar5_yield']:.3f}")

One short run on 18-19 held-out seasons is noisy; the report's numbers average 5 seeds of 300 iterations over all 4 folds. The learning curve of this run:

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, k in zip(ax, ["water", "cvar5_yield", "lambda"]):
    a.plot([c["seasons"] for c in curve], [c[k] for c in curve]); a.set_title(k); a.set_xlabel("training seasons")
plt.tight_layout()

## 5. Case study: the 1967 drought

In [ ]:
display(Image("results/figures/case_study_1967.png", width=850))
display(Image("results/figures/policy_map.png", width=800))